1ª Etapa: Limpeza dos dados

[Link do Dataset no Kaggle](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset)

In [ ]:
import pandas as pd
import ast
import numpy as np

df = pd.read_csv('archive/movies_metadata.csv', low_memory=False)

# Definir todas as colunas JSON que potencialmente existem no arquivo
colunas_json = ['genres', 'belongs_to_collection', 'production_companies', 
                 'production_countries', 'spoken_languages', 'cast', 'crew']

# Lista final das colunas que realmente existem no seu DataFrame
JSON_COLS_TO_PROCESS = [col for col in colunas_json if col in df.columns]
print(f"Colunas JSON que serão processadas: {JSON_COLS_TO_PROCESS}")

Desserialização dos Dados

In [ ]:
# --- 2. FUNÇÃO DE DESSERIALIZAÇÃO SEGURA ---
def desserializar_dados(valor):

    # Padronizar valores vazios "None" e "" como None
    if pd.isna(valor) or valor in ['', 'None']:
        return None
    
    # Executar somente os valores literais do CSV
    try:
        return ast.literal_eval(valor)
    
    # Caso nenhuma das opções acima seja o caso, o valor será vazio
    except:
        return None

for coluna in JSON_COLS_TO_PROCESS:
    # 3.1 Cria a coluna temporária de objeto (ex: df['genres_obj'])
    df[f'{coluna}_obj'] = df[coluna].apply(desserializar_dados)
    
    # 3.2 Preenchimento de nulos para garantir que o objeto seja Lista ou Dicionário
    # (Para evitar erros nos passos de concatenação/contagem)
    if coluna == 'belongs_to_collection':
        # Deve ser Dicionário {} para filmes sem coleção
        df[f'{coluna}_obj'] = df[f'{coluna}_obj'].apply(lambda x: x if isinstance(x, dict) else {})
    elif isinstance(df[f'{col}_obj'].iloc[0], list) or col not in ['belongs_to_collection']: 
        # Deve ser Lista [] para colunas como genres, companies, etc.
        df[f'{coluna}_obj'] = df[f'{coluna}_obj'].apply(lambda x: x if isinstance(x, list) else [])



In [ ]:

# --- 4. TRATAMENTO DA COLUNA 'belongs_to_collection' (Dicionário Único) ---

if 'belongs_to_collection' in JSON_COLS_TO_PROCESS:
    # Extrai o nome da coleção e coloca em uma coluna simples
    df['collection_name'] = df['belongs_to_collection_obj'].apply(
        lambda x: x.get('name') if isinstance(x, dict) else None
    )


# --- 5. NORMALIZAÇÃO DE LISTAS (Contagem e Lista Simples de Nomes) ---

# Colunas que são listas e que foram criadas no passo 3 (excluindo o dicionário 'belongs_to_collection')
LIST_COLS_OBJ = [col for col in JSON_COLS_TO_PROCESS if col != 'belongs_to_collection']

print("--- Criando Contagens e Listas Simples ---")

for col in LIST_COLS_OBJ:
    obj_col = f'{col}_obj' 
    
    # 5.1. Extrai o número de itens na lista (ex: genres_count)
    df[f'{col}_count'] = df[obj_col].apply(lambda x: len(x) if isinstance(x, list) else 0)
    
    # 5.2. Cria uma coluna de texto simples (ex: "Action|Comedy")
    def extract_names(lista):
        if isinstance(lista, list):
            # Extrai o nome de cada dicionário na lista (com tratamento seguro)
            names = [d.get('name') for d in lista if isinstance(d, dict) and d.get('name')]
            return '|'.join(names)
        return ''

    df[f'{col}_list'] = df[obj_col].apply(extract_names)


# --- 6. REMOÇÃO FINAL DAS COLUNAS JSON ORIGINAIS E OBJETOS TEMPORÁRIOS ---

# Colunas originais com JSON/Python object:
cols_to_drop = JSON_COLS_TO_PROCESS

# Colunas temporárias (Python objects) criadas na Etapa 3:
cols_to_drop.extend([f'{col}_obj' for col in JSON_COLS_TO_PROCESS])

# Remove as colunas complexas, preservando apenas as colunas limpas (e as contagens/listas)
df = df.drop(columns=cols_to_drop, errors='ignore')


# --- 7. EXIBIR O RESULTADO FINAL ---
print("\n" + "="*50)
print("✅ LIMPEZA E NORMALIZAÇÃO BÁSICA CONCLUÍDAS.")
print("="*50)

print("\nColunas Restantes no DataFrame Limpo:")
print(df.columns.tolist())

print("\nExemplo de Dados Limpos:")
# Exibe as novas colunas limpas: o nome da coleção e as contagens/listas
print(df[['title', 'collection_name', 'genres_count', 'genres_list', 'production_companies_count']].head())

In [ ]:
df_filtrado = df

In [ ]:
df_filtrado.drop(columns=['homepage', 'imdb_id', 'budget', 'poster_path', 'collection_name', 'production_companies_count', 'production_companies_list', 'production_countries_count'], inplace=True)

In [ ]:
df_filtrado.rename(columns={'adult': 'Adulto', 'original_language': 'Idioma Original', 'original_title': 'Título Original', 'overview': "Descrição", 'popularity': 'Popularidade', 'release_date': 'Data de Lançamento', 'revenue': 'Receita', 'runtime': 'Duração', 'status': 'Status', 'production_countries_list': 'País de Origem', 'spoken_languages_count': 'Idiomas falados no filme', 'spoken_languages_list': 'Lista de Idiomas', 'title': 'Título em inglês'}, inplace=True)

In [ ]:
df_brazil = df_filtrado[df_filtrado['País de Origem'] == 'Brazil']

In [ ]:
df_brazil.sort_values('Data de Lançamento',ascending=False)

In [ ]:
df_filtrado = df_filtrado[['id', 'Título Original', 'Título em inglês', 'País de Origem', 'Idioma Original', 'Idiomas falados no filme', 'Lista de Idiomas', 'genres_list', 'vote_count', 'vote_average', 'Popularidade', 'Duração', 'Data de Lançamento', 'Adulto', 'Status']]

In [ ]:
df_analise = df_filtrado[['Título Original', 'Título em inglês', 'País de Origem', 'Idioma Original']].copy() 

# 2. Cria a coluna booleana 'Analise'
# Verifica se os valores são iguais e armazena o resultado (True/False)
df_analise['Analise'] = df_filtrado['Título Original'] == df_filtrado['Título em inglês']

df_analise.loc[df_analise['Analise'] == False]

In [ ]:
df_analise

Analisando o DataSet Keywords

In [ ]:
import pandas as pd
import ast
import numpy as np

# --- 1. CARREGAR O DATAFRAME ---
# Assumindo que o arquivo keywords.csv está na mesma pasta raiz
try:
    df_keywords = pd.read_csv('archive/keywords.csv')
except FileNotFoundError:
    print("ERRO: Arquivo 'keywords.csv' não encontrado. Verifique o caminho.")
    exit()

# --- 2. FUNÇÃO DE DESSERIALIZAÇÃO SEGURA ---
def safe_literal_eval(val):
    """Converte strings literais (como JSON) em objetos Python, tratando nulos e erros."""
    if pd.isna(val) or val in ['', 'None', '[]']:
        return []
    try:
        return ast.literal_eval(val)
    except:
        return []

# --- 3. APLICAR DESSERIALIZAÇÃO ---
df_keywords['keywords_obj'] = df_keywords['keywords'].apply(safe_literal_eval)

# --- 4. NORMALIZAÇÃO: Achatando a Lista de Dicionários ---

# Achata a coluna 'keywords_obj', criando uma linha para cada palavra-chave por filme.
keywords_normalized = pd.json_normalize(
    df_keywords.to_dict('records'),  # Converte o DF para o formato que json_normalize espera
    record_path='keywords_obj',      # Onde está a lista que queremos achatar
    meta=['id'],                     # Mantém a coluna 'id' do filme como metadado
    record_prefix='keyword_'         # Prefixo para as colunas extraídas (ex: keyword_id, keyword_name)
)

# --- 5. LIMPEZA FINAL ---

# Remove a coluna original e a coluna temporária de objetos
keywords_normalized = keywords_normalized.drop(columns=['keywords_obj'], errors='ignore')


# --- 6. EXIBIR O RESULTADO ---
print("\n" + "="*50)
print("✅ NORMALIZAÇÃO DE KEYWORDS CONCLUÍDA.")
print("="*50)

print("\nExemplo de Dados Normalizados (Palavra-Chave por Linha):")
# Exibe as colunas: ID do filme, ID da keyword e Nome da keyword
print(keywords_normalized[['id', 'keyword_id', 'keyword_name']].head(10))

In [ ]:
keywords_normalized

In [ ]:
df_filtrado.head(5)

In [ ]:
keywords_normalized.head(5)

In [ ]:
keywords_normalized.groupby(by= ['id'])

In [ ]:
import pandas as pd

# 1. Agrupar as palavras-chave por ID de filme
# Usamos o ID do filme ('id') para agrupar, e a função .agg() para juntar
# todos os nomes das palavras-chave ('keyword_name') em uma única string,
# separada por um pipe '|'.

keywords_agregadas = keywords_normalized.groupby('id')['keyword_name'].agg(lambda x: '|'.join(x)).reset_index()

# 2. Renomear a coluna agregada
# A coluna resultante da agregação é nomeada 'keyword_name', renomeamos para ser descritiva
keywords_agregadas.rename(columns={'keyword_name': 'keywords_list'}, inplace=True)

# 3. Juntar (Merge) ao DataFrame principal (df_filtrado)
# O merge é feito usando a coluna 'id', que é comum a ambos os DataFrames.

# A coluna de ID no df_filtrado pode ter sido convertida para string ou float. 
# Para evitar problemas, garantimos que ambas as colunas 'id' sejam tratadas como números inteiros,
# caso ainda não estejam (o que é comum neste dataset de filmes).
try:
    df_filtrado['id'] = pd.to_numeric(df_filtrado['id'], errors='coerce').astype('Int64')
    keywords_agregadas['id'] = pd.to_numeric(keywords_agregadas['id'], errors='coerce').astype('Int64')
except:
    print("Aviso: Falha na conversão de ID para inteiro, usando tipo existente.")


df_filtrado = pd.merge(
    df_filtrado, 
    keywords_agregadas, 
    on='id', 
    how='left' # Usamos 'left' para manter todos os filmes em df_filtrado
)

# 4. Visualizar o Resultado
print("Agregação de Keywords Concluída. Novas colunas:")
print(df_filtrado[['Título Original', 'keywords_list']].head())

In [ ]:
df_filtrado.iloc[0]

In [ ]:
df_filtrado.info()

In [ ]:
df_filtrado.sort_values('Data de Lançamento')

In [ ]:
df_filtrado.dropna(inplace= True)

In [ ]:
df_filtrado.loc[(df_filtrado['País de Origem'] == "") & (df_filtrado['Idioma Original'] == "")]

In [ ]:
df_filtrado.loc[(df_filtrado['País de Origem'] == "")].groupby('Idioma Original').count()

In [ ]:
contagem_por_idioma = df_filtrado.loc[df_filtrado['País de Origem'] == ""].groupby('Idioma Original').size()

print("Contagem de Filmes (sem País de Origem) por Idioma:")
print(contagem_por_idioma)

In [ ]:
import pandas as pd

# 🗺️ Dicionário de Mapeamento Idioma -> País (Simplificado para o Exemplo)
country_map = {
    'en': 'USA',         
    'it': 'Italy',      
    'fr': 'France',     
    'de': 'Germany',   
    'es': 'Spain',     
    'hi': 'India',      
    'ru': 'Russia',    
    'fi': 'Finland',   
    'tr': 'Turkey',    
    'nl': 'Netherlands',
    'ja': 'Japan',      
    'sv': 'Sweden',     
    'pt': 'Brazil',     
    'ko': 'South Korea',
    'zh': 'China',      
    'cn': 'China',      
    'el': 'Greece',     
    'pl': 'Poland',     
    'da': 'Denmark',    
    'ar': 'Egypt',      
    'fa': 'Iran',       
    'hu': 'Hungary',    
    'no': 'Norway',     
    'cs': 'Czechia',    
    'te': 'India',      
    'uk': 'Ukraine',    
    'he': 'Israel',     
    'vi': 'Vietnam',    
    'is': 'Iceland',    
    'et': 'Estonia',    
    'cy': 'UK',         
    'sq': 'Albania',    
    'ml': 'India',      
    'mr': 'India',      
    'bn': 'Bangladesh', 
    'ur': 'Pakistan',   
    'uz': 'Uzbekistan', 
    'ab': 'Georgia',    
    'ka': 'Georgia',    
    'eu': 'Spain',      
    'kn': 'India',      
    'ta': 'India',      
    'fy': 'Netherlands',
    'nb': 'Norway',     
    'xx': 'N/A'         
}

# --- 1. Criar o filtro booleano para as linhas sem País de Origem ---
filtro_sem_pais = df_filtrado['País de Origem'] == ""

# --- 2. Aplicar o mapeamento (map) somente às linhas filtradas ---

# 2.1. Seleciona a coluna 'Idioma Original' SOMENTE para as linhas filtradas.
# 2.2. Aplica o dicionário country_map a esses valores de idioma.
# 2.3. O resultado é a nova Série de países (ex: 'USA', 'France', 'Brazil').
novos_paises = df_filtrado.loc[filtro_sem_pais, 'Idioma Original'].map(country_map)

# --- 3. Atribuir os novos valores de volta à coluna 'País de Origem' ---

# Usamos .loc novamente para ATRIBUIR os novos valores APENAS às linhas filtradas
df_filtrado.loc[filtro_sem_pais, 'País de Origem'] = novos_paises

# 4. Visualização de uma amostra para confirmar a imputação
print("✅ Imputação de País de Origem com base no Idioma concluída.")
print("\nExemplo de Filmes Onde o País foi Preenchido:")
# Filtra novamente as linhas que estavam vazias e que agora foram preenchidas
# (Note que algumas podem continuar vazias se o idioma original não estava no dicionário)
print(df_filtrado.loc[df_filtrado['País de Origem'] != ""].head(10))

In [ ]:
df_filtrado.sort_values('Duração')

In [ ]:
df_filtrado

In [ ]:
df_filtrado.to_csv('df_consolidado.csv')

In [ ]:
def avaliar_duracao(duracao):
    if duracao > 180:
        conclusao = 'Very Long'
    elif 120 <= duracao <= 180:
         conclusao = 'Long'
    elif 60 <= duracao < 120:
         conclusao = 'Medium'
    elif 30 <= duracao < 60:
         conclusao = 'Short'
    elif duracao < 30:
         conclusao = 'Very Short'

    return conclusao

In [ ]:
df_filtrado['Categoria de Tempo'] = df_filtrado['Duração'].apply(avaliar_duracao)

In [ ]:
df_filtrado.to_csv('df_consolidado.csv')